# diffcast — full VAE fine-tune + EDM latent diffusion forecaster (INSAT-3S)

Parallel pipeline to `flowcast/`. Trains the FlowCast AutoencoderKL end-to-end on INSAT data (encoder + decoder both unfrozen), then trains an EDM latent diffusion model on the new latents to predict 48-frame day-ahead forecasts.

**Order of cells (per channel — VIS first, then WV):**

1. Train VAE — Phase 1, target sanity PSNR ≥ 30 dB (~3-6 h on T4)
2. Uncomment `vae.full_ckpt` in the config (sed cell)
3. Sanity check + encode new latents — Phase 2 (~15 min)
4. Train EDM diffusion — Phase 4 (~8-12 h on T4)
5. Forecast with --samples 8 — Phase 5 (~5 min)
6. Inspect metrics + grids

Set runtime to **T4 GPU** (or A100/H100 for faster). `flowcast/` is NOT touched by this notebook.

## Setup — mount Drive, cd, install deps (run after every fresh runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO/ISRO A.1'   # <-- edit if your Drive path differs
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q lpips==0.1.4 imagecodecs huggingface_hub
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## VIS — Phase 1: full VAE fine-tune (encoder + decoder)

Loads SEVIR pretrained → unfreezes everything → trains 15,000 steps with `1.0·MSE + 0.5·LPIPS-VGG + 1e-6·KL`. Saves best-by-val-PSNR to `vis/checkpoints_diff/vae_full.pt`. Progress PNGs at `vis/outputs_diff/vae_train_progress/`.

**Gate: val PSNR ≥ 30 dB.** If below, extend with `--steps 25000`.

In [ ]:
!python -m diffcast.train_vae --config vis/config_diff.yaml

## VIS — Phase 2: enable the fine-tuned VAE in config + sanity check + encode latents

Uncomments `vae.full_ckpt: vis/checkpoints_diff/vae_full.pt` in the config, runs sanity_check (gate ≥ 30 dB), then encodes all VIS TIFFs to NEW latents at `vis/latents_diff/`.

In [ ]:
# Enable the fine-tuned VAE for inference
import re, pathlib
cfg_path = pathlib.Path('vis/config_diff.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*full_ckpt:\s*vis/checkpoints_diff/vae_full\.pt.*$',
    r'\1full_ckpt: vis/checkpoints_diff/vae_full.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'vae' in line.lower() or 'full_ckpt' in line:
        print(' ', line)

In [ ]:
!python -m diffcast.sanity_check --config vis/config_diff.yaml --num 30
!python -m diffcast.encode_latents --config vis/config_diff.yaml

## VIS — Phase 4: train EDM latent diffusion forecaster (~8-12 h on T4)

Same Earthformer-UNet backbone flowcast uses, wrapped with EDM preconditioning + Karras-schedule sampling. 100 epochs, fp16, EMA. Best-EMA checkpoint at `vis/checkpoints_diff/best_diff.pt`.

In [ ]:
!python -m diffcast.train_diffusion --config vis/config_diff.yaml

## VIS — Phase 5: 48-frame forecast (3-way ensemble: pixmean / medoid / latmean)

Heun ODE sampling (18 steps per autoregressive block × 4 blocks = 48 frames), 8-sample ensemble, decoded through the fine-tuned VAE.

In [ ]:
!python -m diffcast.forecast_diff --config vis/config_diff.yaml --samples 8

In [ ]:
# Display metrics + grids
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_grid.png'))[-3:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_metrics.json')):
    m = json.load(open(p))
    print(f"\n{p}  samples={m.get('samples')}  medoid_idx={m.get('medoid_idx')}  "
          f"sampler_steps={m.get('sampler_steps')}")
    print(f"  thresholds: {m.get('thresholds')}")
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        print(f"  [{mode}]")
        for k, v in (smry or {}).items():
            print(f"    {k:>10}: {v:.4f}")
    if m.get('crps_mean') is not None:
        print(f"  [crps]\n    CRPS: {m['crps_mean']:.4f}")

## VIS — Phase 6: comparison vs flowcast

Side-by-side: VIS diffcast vs VIS flowcast (baseline / A / B). Same forecast date, same metrics.

In [ ]:
# Compare diffcast vs flowcast on the same forecast date
import glob, json, os

print(f"{'pipeline':<30s} {'mode':<10s} {'psnr':>7s} {'ssim':>7s} {'CSI_M':>7s}  {'CRPS':>8s}")
print('-' * 75)

# flowcast metrics (all live in vis/outputs_fc/)
for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_ens8_metrics*.json')):
    m = json.load(open(p))
    label = ('flowcast/' + os.path.basename(p)
             .replace('flow_forecast_', '')
             .replace('_metrics.json', '')
             .replace('_metrics_', '_'))[:30]
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        crps = m.get('crps_mean')
        crps_str = f"{crps:.4f}" if crps is not None else 'n/a'
        print(f"{label:<30s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}  {crps_str:>8s}")

# diffcast metrics
for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_ens8_metrics.json')):
    m = json.load(open(p))
    label = ('diffcast/' + os.path.basename(p)
             .replace('diff_forecast_', '')
             .replace('_metrics.json', ''))[:30]
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        crps = m.get('crps_mean')
        crps_str = f"{crps:.4f}" if crps is not None else 'n/a'
        print(f"{label:<30s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}  {crps_str:>8s}")

## WV — Phases 1 → 5 (same recipe, swap config)

Run only after VIS validates. WV is 24h continuous so no day/night handling; expect smaller gains than VIS (the SEVIR VAE was already in-distribution for WV per the flowcast experiments).

In [ ]:
!python -m diffcast.train_vae --config wv/config_diff.yaml

In [ ]:
import re, pathlib
cfg_path = pathlib.Path('wv/config_diff.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*full_ckpt:\s*wv/checkpoints_diff/vae_full\.pt.*$',
    r'\1full_ckpt: wv/checkpoints_diff/vae_full.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'full_ckpt' in line: print(' ', line)

In [ ]:
!python -m diffcast.sanity_check --config wv/config_diff.yaml --num 30
!python -m diffcast.encode_latents --config wv/config_diff.yaml

In [ ]:
!python -m diffcast.train_diffusion --config wv/config_diff.yaml

In [ ]:
!python -m diffcast.forecast_diff --config wv/config_diff.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display
for p in sorted(glob.glob('wv/outputs_diff/diff_forecast_*_grid.png'))[-3:]:
    print(p); display(Image(p))
for p in sorted(glob.glob('wv/outputs_diff/diff_forecast_*_metrics.json')):
    m = json.load(open(p))
    print(f"\n{p}  samples={m.get('samples')}")
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        print(f"  [{mode}]")
        for k, v in (smry or {}).items():
            print(f"    {k:>10}: {v:.4f}")
    if m.get('crps_mean') is not None:
        print(f"  [crps] CRPS: {m['crps_mean']:.4f}")

## Snapshot (both channels)

Bundles all diffcast outputs for download. Mirrors the flowcast snapshot pattern; uses `_diff` suffix so it doesn't collide with existing flowcast snapshots.

In [ ]:
import os, shutil, glob, json, zipfile, datetime as dt
import yaml

TS = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

def snapshot_channel(CH):
    SNAP = f'{CH}/results_snapshot_diff_{TS}'
    os.makedirs(SNAP, exist_ok=True)
    for p in glob.glob(f'{CH}/outputs_diff/*'):
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(SNAP, os.path.basename(p)))
    if os.path.isdir(f'{CH}/outputs_diff/vae_train_progress'):
        shutil.copytree(f'{CH}/outputs_diff/vae_train_progress',
                        os.path.join(SNAP, 'vae_train_progress'),
                        dirs_exist_ok=True)
    for src in glob.glob(f'{CH}/checkpoints_diff/*.pt') + \
                glob.glob(f'{CH}/checkpoints_diff/*.json'):
        shutil.copy2(src, os.path.join(SNAP, os.path.basename(src)))
    shutil.copy2(f'{CH}/config_diff.yaml', os.path.join(SNAP, 'config_diff.yaml'))

    zip_path = f'{CH}/results_snapshot_diff_{TS}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(SNAP):
            for f in files:
                full = os.path.join(root, f)
                arc = os.path.relpath(full, os.path.dirname(SNAP))
                zf.write(full, arcname=arc)
    sz = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'[{CH}] snapshot: {SNAP}  zip: {zip_path}  ({sz:.1f} MB)')

for CH in ('vis', 'wv'):
    if os.path.isdir(f'{CH}/outputs_diff'):
        snapshot_channel(CH)
    else:
        print(f'[{CH}] skipped — no outputs_diff (channel not yet run)')